In [1]:
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [2]:
# ============================================================
# PROJECT PATHS
# ============================================================

project_root = Path.cwd().parent

feature_dir = (
    project_root
    / "data"
    / "processed"
    / "feature_engineered"
)

model_dir = (
    project_root
    / "models"
)

recommendation_dir = (
    project_root
    / "data"
    / "processed"
    / "recommendation"
)

recommendation_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [4]:
products = pd.read_csv(
    feature_dir / "products_features.csv"
)

reviews = pd.read_csv(
    feature_dir / "reviews_features.csv"
)

print("Products :", products.shape)
print("Reviews  :", reviews.shape)

C:\Users\ujjwa\AppData\Local\Temp\ipykernel_25000\3662470709.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv(


Products : (8494, 32)
Reviews  : (1088891, 24)


In [5]:
products.head()

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,variation_desc,ingredients,price_usd,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,log_price,discount_amount,discount_pct,popularity_score,text_blob
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,Unknown,Unknown,Unknown,NaN,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD A...",35.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN,3.583519,NaN,NaN,6.244942,Fragrance Discovery Set 19-69 ['Unisex/ Gender...
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0,5.278115,NaN,NaN,6.005682,La Habana Eau de Parfum 19-69 ['Unisex/ Gender...
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0,5.278115,NaN,NaN,5.985870,Rainbow Bar Eau de Parfum 19-69 ['Unisex/ Gend...
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0,5.278115,NaN,NaN,6.044026,Kasbah Eau de Parfum 19-69 ['Unisex/ Genderles...
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,NaN,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,NaN,NaN,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0,5.278115,NaN,NaN,5.794447,Purple Haze Eau de Parfum 19-69 ['Unisex/ Gend...


In [6]:
reviews.head()

,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd,positive_signal,engagement_score,review_length,recency_days,time_weight,interaction_strength
0,538863,1,0.0,NaN,0,0,0,2018-11-01,One use and into the trash this went. I woke u...,one and done,fair,blue,combination,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,0,0,177,1601,0.000624,0.000624
1,549704,5,NaN,1.0,3,0,3,2011-04-21,I like this a lot. I was a little afraid to tr...,Sweet!,lightMedium,Unknown,Unknown,Unknown,P218700,100 percent Pure Argan Oil,Josie Maran,49.0,1,3,284,4352,0.000230,0.004595
2,557770,5,NaN,1.0,1,0,1,2016-02-19,It is ugly and smelly - but this product works...,Sulfur Rules!,Unknown,Unknown,combination,Unknown,P232903,EradiKate Acne Treatment,Kate Somerville,28.0,1,1,254,2587,0.000386,0.003864
3,562130,5,NaN,1.0,3,0,3,2014-11-29,Can’t go wrong with this basic moisturizer. I ...,Good Basic Moisterizer,Unknown,Unknown,dry,Unknown,P381030,Dramatically Different Moisturizing Lotion+,CLINIQUE,32.5,1,3,165,3034,0.000329,0.006590
4,582399,5,1.0,NaN,0,0,0,2018-11-21,I got a sample of this in the k beauty play se...,NaN,fair,hazel,combination,brown,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,1,0,256,1581,0.000632,0.003161


In [7]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8494 entries, 0 to 8493
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_id          8494 non-null   object 
 1   product_name        8494 non-null   object 
 2   brand_id            8494 non-null   int64  
 3   brand_name          8494 non-null   object 
 4   loves_count         8494 non-null   int64  
 5   rating              8216 non-null   float64
 6   reviews             8216 non-null   float64
 7   size                8494 non-null   object 
 8   variation_type      8494 non-null   object 
 9   variation_value     8494 non-null   object 
 10  variation_desc      1250 non-null   object 
 11  ingredients         7549 non-null   object 
 12  price_usd           8494 non-null   float64
 13  value_price_usd     451 non-null    float64
 14  sale_price_usd      270 non-null    float64
 15  limited_edition     8494 non-null   int64  
 16  new   

In [8]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1088891 entries, 0 to 1088890
Data columns (total 24 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   author_id                 1088891 non-null  object 
 1   rating                    1088891 non-null  int64  
 2   is_recommended            924416 non-null   float64
 3   helpfulness               530535 non-null   float64
 4   total_feedback_count      1088891 non-null  int64  
 5   total_neg_feedback_count  1088891 non-null  int64  
 6   total_pos_feedback_count  1088891 non-null  int64  
 7   submission_time           1088891 non-null  object 
 8   review_text               1087450 non-null  object 
 9   review_title              779152 non-null   object 
 10  skin_tone                 1088891 non-null  object 
 11  eye_color                 1088891 non-null  object 
 12  skin_type                 1088891 non-null  object 
 13  hair_color                1

In [9]:
products.isna().sum()

product_id               0
product_name             0
brand_id                 0
brand_name               0
loves_count              0
rating                 278
reviews                278
size                     0
variation_type           0
variation_value          0
variation_desc        7244
ingredients            945
price_usd                0
value_price_usd       8043
sale_price_usd        8224
limited_edition          0
new                      0
online_only              0
out_of_stock             0
sephora_exclusive        0
highlights            2207
primary_category         0
secondary_category       0
tertiary_category        0
child_count              0
child_max_price       5740
child_min_price       5740
log_price                0
discount_amount       8459
discount_pct          8459
popularity_score       278
text_blob                0
dtype: int64

In [10]:
reviews.isna().sum()

author_id                        0
rating                           0
is_recommended              164475
helpfulness                 558356
total_feedback_count             0
total_neg_feedback_count         0
total_pos_feedback_count         0
submission_time                  0
review_text                   1441
review_title                309739
skin_tone                        0
eye_color                        0
skin_type                        0
hair_color                       0
product_id                       0
product_name                     0
brand_name                       0
price_usd                        0
positive_signal                  0
engagement_score                 0
review_length                    0
recency_days                     0
time_weight                      0
interaction_strength             0
dtype: int64

In [16]:
products[
    [
        "rating",
        "reviews",
        "price_usd",
        "value_price_usd",
        "sale_price_usd",
        "child_max_price",
        "child_min_price",
        "discount_amount",
        "discount_pct"
    ]
].describe()

,rating,reviews,price_usd,value_price_usd,sale_price_usd,child_max_price,child_min_price,discount_amount,discount_pct
count,8494.000000,8494.000000,8494.000000,451.000000,8494.000000,8494.000000,8494.000000,8494.000000,8494.000000
mean,4.197617,433.865081,51.655595,91.168537,51.231958,50.393619,45.813491,0.180221,0.246328
std,0.508448,1086.731772,53.669234,79.195631,53.781548,54.943749,48.960179,3.716254,3.965262
min,1.000000,0.000000,3.000000,0.000000,1.750000,3.000000,3.000000,0.000000,0.000000
25%,4.000000,22.000000,25.000000,45.000000,25.000000,24.000000,22.000000,0.000000,0.000000
50%,4.289350,112.000000,35.000000,67.000000,35.000000,34.000000,32.000000,0.000000,0.000000
75%,4.522500,402.000000,58.000000,108.500000,58.000000,56.000000,50.000000,0.000000,0.000000
max,5.000000,21281.000000,1900.000000,617.000000,1900.000000,1900.000000,1900.000000,160.000000,85.000000


In [12]:
text_columns = [
    "variation_desc",
    "ingredients",
    "highlights"
]

products[text_columns].head()

,variation_desc,ingredients,highlights
0,Unknown,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD A...","['Unisex/ Genderless Scent', 'Warm &Spicy Scen..."
1,Unknown,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...","['Unisex/ Genderless Scent', 'Layerable Scent'..."
2,Unknown,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...","['Unisex/ Genderless Scent', 'Layerable Scent'..."
3,Unknown,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...","['Unisex/ Genderless Scent', 'Layerable Scent'..."
4,Unknown,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...","['Unisex/ Genderless Scent', 'Layerable Scent'..."


In [13]:
for col in text_columns:
    print(f"{col}: {products[col].nunique()}")

variation_desc: 936
ingredients: 6539
highlights: 4418


In [14]:
reviews[
    [
        "is_recommended",
        "helpfulness",
        "review_text",
        "review_title"
    ]
].head()

,is_recommended,helpfulness,review_text,review_title
0,0.0,NaN,One use and into the trash this went. I woke u...,one and done
1,NaN,1.0,I like this a lot. I was a little afraid to tr...,Sweet!
2,NaN,1.0,It is ugly and smelly - but this product works...,Sulfur Rules!
3,NaN,1.0,Can’t go wrong with this basic moisturizer. I ...,Good Basic Moisterizer
4,1.0,NaN,I got a sample of this in the k beauty play se...,NaN


In [15]:
reviews[
    [
        "is_recommended",
        "helpfulness"
    ]
].describe()

,is_recommended,helpfulness
count,924416.000000,530535.000000
mean,0.839947,0.767873
std,0.366656,0.316987
min,0.000000,0.000000
25%,1.000000,0.653846
50%,1.000000,0.928571
75%,1.000000,1.000000
max,1.000000,1.000000


In [17]:
products.drop(
    columns=["variation_desc"],
    inplace=True
)

In [18]:
products.drop(
    columns=[
        "discount_amount",
        "discount_pct"
    ],
    inplace=True
)

In [19]:
products["value_price_usd"] = (
    products["value_price_usd"]
    .fillna(products["price_usd"])
)

In [20]:
products["sale_price_usd"] = (
    products["sale_price_usd"]
    .fillna(products["price_usd"])
)

In [21]:
products["child_max_price"] = (
    products["child_max_price"]
    .fillna(products["price_usd"])
)

products["child_min_price"] = (
    products["child_min_price"]
    .fillna(products["price_usd"])
)

In [22]:
products["rating"] = (
    products["rating"]
    .fillna(products["rating"].median())
)

In [23]:
products["reviews"] = (
    products["reviews"]
    .fillna(0)
)

In [24]:
products["popularity_score"] = (
    0.6 * np.log1p(products["loves_count"])
    +
    0.3 * products["rating"]
    +
    0.1 * np.log1p(products["reviews"])
)

In [25]:
products["ingredients"] = (
    products["ingredients"]
    .fillna("Unknown")
)

In [26]:
products["highlights"] = (
    products["highlights"]
    .fillna("Unknown")
)

In [27]:
products.isna().sum()

product_id            0
product_name          0
brand_id              0
brand_name            0
loves_count           0
rating                0
reviews               0
size                  0
variation_type        0
variation_value       0
ingredients           0
price_usd             0
value_price_usd       0
sale_price_usd        0
limited_edition       0
new                   0
online_only           0
out_of_stock          0
sephora_exclusive     0
highlights            0
primary_category      0
secondary_category    0
tertiary_category     0
child_count           0
child_max_price       0
child_min_price       0
log_price             0
popularity_score      0
text_blob             0
dtype: int64

In [28]:
reviews["review_text"] = (
    reviews["review_text"]
    .fillna("")
)

In [29]:
reviews["review_title"] = (
    reviews["review_title"]
    .fillna("")
)

In [30]:
reviews["is_recommended"] = (
    reviews["is_recommended"]
    .fillna(
        (reviews["rating"] >= 4).astype(int)
    )
)

In [31]:
reviews["helpfulness"] = (
    reviews["helpfulness"]
    .fillna(0.5)
)

In [32]:
reviews.isna().sum()

author_id                   0
rating                      0
is_recommended              0
helpfulness                 0
total_feedback_count        0
total_neg_feedback_count    0
total_pos_feedback_count    0
submission_time             0
review_text                 0
review_title                0
skin_tone                   0
eye_color                   0
skin_type                   0
hair_color                  0
product_id                  0
product_name                0
brand_name                  0
price_usd                   0
positive_signal             0
engagement_score            0
review_length               0
recency_days                0
time_weight                 0
interaction_strength        0
dtype: int64

In [33]:
review_features = (
    reviews
    .groupby("product_id")
    .agg(
        avg_rating=("rating", "mean"),
        total_reviews=("rating", "count"),
        recommendation_rate=("is_recommended", "mean"),
        avg_helpfulness=("helpfulness", "mean"),
        avg_interaction=("interaction_strength", "mean"),
        avg_review_length=("review_length", "mean")
    )
    .reset_index()
)

review_features.head()

,product_id,avg_rating,total_reviews,recommendation_rate,avg_helpfulness,avg_interaction,avg_review_length
0,P107306,4.028226,248,0.762097,0.743024,0.014109,293.596774
1,P114902,4.415655,1482,0.866397,0.657307,0.008805,390.387989
2,P12045,4.442515,1670,0.873653,0.645504,0.018810,259.376048
3,P122651,4.515306,196,0.887755,0.657226,0.011238,222.076531
4,P122661,4.520725,772,0.898964,0.613192,0.003952,239.492228


In [34]:
products = products.merge(
    review_features,
    on="product_id",
    how="left"
)

products.head()

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,ingredients,price_usd,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,log_price,popularity_score,text_blob,avg_rating,total_reviews,recommendation_rate,avg_helpfulness,avg_interaction,avg_review_length
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,Unknown,Unknown,Unknown,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD A...",35.0,35.0,35.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,35.0,35.0,3.583519,6.590390,Fragrance Discovery Set 19-69 ['Unisex/ Gender...,NaN,NaN,NaN,NaN,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,195.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0,5.278115,6.460104,La Habana Eau de Parfum 19-69 ['Unisex/ Gender...,NaN,NaN,NaN,NaN,NaN,NaN
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,195.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0,5.278115,6.410906,Rainbow Bar Eau de Parfum 19-69 ['Unisex/ Gend...,NaN,NaN,NaN,NaN,NaN,NaN
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,195.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0,5.278115,6.459573,Kasbah Eau de Parfum 19-69 ['Unisex/ Genderles...,NaN,NaN,NaN,NaN,NaN,NaN
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,195.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0,5.278115,5.971970,Purple Haze Eau de Parfum 19-69 ['Unisex/ Gend...,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
products = products.merge(
    review_features,
    on="product_id",
    how="left"
)

products.head()

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,ingredients,price_usd,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,log_price,popularity_score,text_blob,avg_rating_x,total_reviews_x,recommendation_rate_x,avg_helpfulness_x,avg_interaction_x,avg_review_length_x,avg_rating_y,total_reviews_y,recommendation_rate_y,avg_helpfulness_y,avg_interaction_y,avg_review_length_y
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,Unknown,Unknown,Unknown,"['Capri Eau de Parfum:', 'Alcohol Denat. (SD A...",35.0,35.0,35.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,35.0,35.0,3.583519,6.590390,Fragrance Discovery Set 19-69 ['Unisex/ Gender...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,195.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0,5.278115,6.460104,La Habana Eau de Parfum 19-69 ['Unisex/ Gender...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,195.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0,5.278115,6.410906,Rainbow Bar Eau de Parfum 19-69 ['Unisex/ Gend...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,195.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0,5.278115,6.459573,Kasbah Eau de Parfum 19-69 ['Unisex/ Genderles...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,"['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...",195.0,195.0,195.0,0,0,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0,5.278115,5.971970,Purple Haze Eau de Parfum 19-69 ['Unisex/ Gend...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
review_features.head()

,product_id,avg_rating,total_reviews,recommendation_rate,avg_helpfulness,avg_interaction,avg_review_length
0,P107306,4.028226,248,0.762097,0.743024,0.014109,293.596774
1,P114902,4.415655,1482,0.866397,0.657307,0.008805,390.387989
2,P12045,4.442515,1670,0.873653,0.645504,0.018810,259.376048
3,P122651,4.515306,196,0.887755,0.657226,0.011238,222.076531
4,P122661,4.520725,772,0.898964,0.613192,0.003952,239.492228


In [42]:
review_features.columns

Index(['product_id', 'avg_rating', 'total_reviews', 'recommendation_rate',
       'avg_helpfulness', 'avg_interaction', 'avg_review_length'],
      dtype='object')

In [43]:
products.columns

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd',
       'limited_edition', 'new', 'online_only', 'out_of_stock',
       'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price', 'log_price', 'popularity_score',
       'text_blob', 'avg_rating_x', 'total_reviews_x', 'recommendation_rate_x',
       'avg_helpfulness_x', 'avg_interaction_x', 'avg_review_length_x',
       'avg_rating_y', 'total_reviews_y', 'recommendation_rate_y',
       'avg_helpfulness_y', 'avg_interaction_y', 'avg_review_length_y'],
      dtype='object')

In [44]:
products.drop(
    columns=[
        "avg_rating_x",
        "total_reviews_x",
        "recommendation_rate_x",
        "avg_helpfulness_x",
        "avg_interaction_x",
        "avg_review_length_x"
    ],
    inplace=True
)

In [45]:
products.rename(
    columns={
        "avg_rating_y": "avg_rating",
        "total_reviews_y": "total_reviews",
        "recommendation_rate_y": "recommendation_rate",
        "avg_helpfulness_y": "avg_helpfulness",
        "avg_interaction_y": "avg_interaction",
        "avg_review_length_y": "avg_review_length"
    },
    inplace=True
)

In [46]:
products.columns

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd',
       'limited_edition', 'new', 'online_only', 'out_of_stock',
       'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price', 'log_price', 'popularity_score',
       'text_blob', 'avg_rating', 'total_reviews', 'recommendation_rate',
       'avg_helpfulness', 'avg_interaction', 'avg_review_length'],
      dtype='object')

In [47]:
cols = [
    "avg_rating",
    "total_reviews",
    "recommendation_rate",
    "avg_helpfulness",
    "avg_interaction",
    "avg_review_length"
]

products[cols] = products[cols].fillna(0)

In [48]:
products[cols].isna().sum()

avg_rating             0
total_reviews          0
recommendation_rate    0
avg_helpfulness        0
avg_interaction        0
avg_review_length      0
dtype: int64

In [49]:
products["log_loves"]=np.log1p(products["loves_count"])
products["log_reviews"]=np.log1p(products["total_reviews"])

In [50]:
products["price_bucket"] = pd.cut(
    products["price_usd"],
    bins=[0, 25, 50, 100, np.inf],
    labels=[
        "Budget",
        "Affordable",
        "Premium",
        "Luxury"
    ]
)

In [51]:
products["price_bucket"].value_counts()

price_bucket
Affordable    3707
Budget        2314
Premium       1575
Luxury         898
Name: count, dtype: int64

Wilson Score

In [52]:
from math import sqrt

def wilson_score(pos, n, z=1.96):
    if n == 0:
        return 0

    phat = pos / n

    return (
        (
            phat
            + z * z / (2 * n)
            - z * sqrt(
                (
                    phat * (1 - phat)
                    + z * z / (4 * n)
                ) / n
            )
        )
        /
        (1 + z * z / n)
    )

In [53]:
products["wilson_score"] = products.apply(
    lambda row: wilson_score(
        row["recommendation_rate"] * row["total_reviews"],
        row["total_reviews"]
    ),
    axis=1
)

In [54]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

score_cols = [
    "log_loves",
    "avg_rating",
    "log_reviews",
    "recommendation_rate",
    "wilson_score",
    "avg_helpfulness",
    "avg_interaction"
]

products[[c + "_scaled" for c in score_cols]] = scaler.fit_transform(
    products[score_cols]
)

In [55]:
products.filter(like="_scaled").head()

,log_loves_scaled,avg_rating_scaled,log_reviews_scaled,recommendation_rate_scaled,wilson_score_scaled,avg_helpfulness_scaled,avg_interaction_scaled
0,0.618370,0.0,0.0,0.0,0.0,0.0,0.0
1,0.582933,0.0,0.0,0.0,0.0,0.0,0.0
2,0.571454,0.0,0.0,0.0,0.0,0.0,0.0
3,0.566157,0.0,0.0,0.0,0.0,0.0,0.0
4,0.558057,0.0,0.0,0.0,0.0,0.0,0.0


In [56]:
products["final_popularity_score"] = (
      0.30 * products["log_loves_scaled"]
    + 0.20 * products["avg_rating_scaled"]
    + 0.15 * products["log_reviews_scaled"]
    + 0.15 * products["recommendation_rate_scaled"]
    + 0.10 * products["wilson_score_scaled"]
    + 0.05 * products["avg_helpfulness_scaled"]
    + 0.05 * products["avg_interaction_scaled"]
)

In [57]:
popular_products = (
    products
    .sort_values(
        by="final_popularity_score",
        ascending=False
    )
    .reset_index(drop=True)
)

popular_products.head(10)

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,ingredients,price_usd,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price,log_price,popularity_score,text_blob,avg_rating,total_reviews,recommendation_rate,avg_helpfulness,avg_interaction,avg_review_length,log_loves,log_reviews,price_bucket,wilson_score,log_loves_scaled,avg_rating_scaled,log_reviews_scaled,recommendation_rate_scaled,wilson_score_scaled,avg_helpfulness_scaled,avg_interaction_scaled,final_popularity_score
0,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,6125,LANEIGE,1081315,4.3508,16118.0,0.7 oz/ 20 g,Color,Original,"['Diisostearyl Malate, Hydrogenated Polyisobut...",24.0,24.0,24.0,0,0,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,Unknown,3,24.0,24.0,3.218876,10.610229,Lip Sleeping Mask Intense Hydration with Vitam...,4.347568,15853.0,0.830316,0.610540,0.025325,275.525326,13.893689,9.671177,Budget,0.824393,0.981696,0.869514,1.000000,0.830316,0.844549,0.610540,0.003358,0.858109
1,P442563,AHA 30% + BHA 2% Exfoliating Peeling Solution,6234,The Ordinary,533877,4.5734,3242.0,1 oz/ 30 mL,Size,1 oz/ 30 mL,"['Glycolic Acid, Aqua (Water), Aloe Barbadensi...",9.5,9.5,9.5,0,0,0,0,0,"['Vegan', 'Community Favorite', 'AHA/Glycolic ...",Skincare,Treatments,Facial Peels,0,9.5,9.5,2.351375,10.093199,AHA 30% + BHA 2% Exfoliating Peeling Solution ...,4.572046,3241.0,0.916384,0.744268,0.091251,321.155508,13.187923,8.083946,Budget,0.906354,0.931828,0.914409,0.835880,0.916384,0.928513,0.744268,0.012101,0.855940
2,P269122,Alpha Beta Extra Strength Daily Peel Pads,5668,Dr. Dennis Gross Skincare,234295,4.5455,7412.0,30 Treatments + 5 Bonus,Size,30 Treatments + 5 Bonus,"['Step 1:', 'Water/Aqua/Eau, Alcohol Denat. (S...",92.0,102.0,92.0,0,0,0,0,1,"['Good for: Dullness/Uneven Texture', 'Clean a...",Skincare,Treatments,Facial Peels,2,153.0,20.0,4.532599,9.673353,Alpha Beta Extra Strength Daily Peel Pads Dr. ...,4.543779,7378.0,0.930062,0.594313,0.018567,293.744917,12.364341,8.906393,Premium,0.924016,0.873635,0.908756,0.920921,0.930062,0.946607,0.594313,0.002462,0.845989
3,P417238,Green Clean Makeup Removing Cleansing Balm,8001,Farmacy,403801,4.4958,6158.0,3.4 oz/ 100 mL,Size,3.4 oz/ 100 mL,"['Cetyl Ethylhexanoate, Caprylic/Capric Trigly...",36.0,36.0,36.0,0,0,0,0,1,"['Vegan', 'Good for: Dullness/Uneven Texture',...",Skincare,Cleansers,Makeup Removers,2,60.0,24.0,3.610918,9.966515,Green Clean Makeup Removing Cleansing Balm Far...,4.496422,6148.0,0.875569,0.628400,0.019890,293.619063,12.908680,8.724045,Affordable,0.867083,0.912097,0.899284,0.902066,0.875569,0.888283,0.628400,0.002638,0.840512
4,P248407,Ultra Repair Cream Intense Hydration,5972,First Aid Beauty,300432,4.5200,7539.0,6 oz/ 170 g,Size,6 oz/ 170 g,"['Colloidal Oatmeal 0.50%, Water, Stearic Acid...",38.0,38.0,38.0,0,0,0,0,0,"['Best for Dry Skin', 'Community Favorite', 'C...",Skincare,Moisturizers,Moisturizers,3,48.0,18.0,3.663562,9.816586,Ultra Repair Cream Intense Hydration First Aid...,4.512119,7385.0,0.879621,0.640138,0.014438,330.119567,12.612980,8.907342,Affordable,0.872001,0.891204,0.902424,0.921019,0.879621,0.893321,0.640138,0.001915,0.839377
5,P173726,Facial Cotton,5337,Shiseido,133163,4.8284,2827.0,165 sheets,Size,165 sheets,['100% Cotton.'],13.0,13.0,13.0,0,0,0,0,0,Unknown,Skincare,Cleansers,Makeup Removers,1,5.0,5.0,2.639057,9.322855,Facial Cotton Shiseido ['100% Cotton.'],4.826071,2777.0,0.962550,0.591526,0.009593,225.371624,11.799337,7.929487,Budget,0.954825,0.833714,0.965214,0.819909,0.962550,0.978169,0.591526,0.001272,0.837983
6,P427417,Niacinamide 10% + Zinc 1% Oil Control Serum,6234,The Ordinary,763168,4.2439,5778.0,1 oz/ 30 mL,Size,1 oz/ 30 mL,"['Aqua (Water), Niacinamide, Pentylene Glycol,...",6.0,6.0,6.0,0,0,0,0,0,"['Vegan', 'Community Favorite', 'Oil Fr

In [58]:
def recommend_popular(
    category=None,
    brand=None,
    price_bucket=None,
    top_n=10
):
    
    recommendations = popular_products.copy()
    
    
    if category is not None:
        recommendations = recommendations[
            recommendations["primary_category"]
            .str.lower()
            ==
            category.lower()
        ]
    
    
    if brand is not None:
        recommendations = recommendations[
            recommendations["brand_name"]
            .str.lower()
            ==
            brand.lower()
        ]
    
    
    if price_bucket is not None:
        recommendations = recommendations[
            recommendations["price_bucket"]
            ==
            price_bucket
        ]
    
    
    return (
        recommendations[
            [
                "product_name",
                "brand_name",
                "primary_category",
                "price_usd",
                "rating",
                "reviews",
                "loves_count",
                "popularity_score"
            ]
        ]
        .head(top_n)
    )

In [59]:
recommend_popular()

,product_name,brand_name,primary_category,price_usd,rating,reviews,loves_count,popularity_score
0,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Skincare,24.0,4.3508,16118.0,1081315,10.610229
1,AHA 30% + BHA 2% Exfoliating Peeling Solution,The Ordinary,Skincare,9.5,4.5734,3242.0,533877,10.093199
2,Alpha Beta Extra Strength Daily Peel Pads,Dr. Dennis Gross Skincare,Skincare,92.0,4.5455,7412.0,234295,9.673353
3,Green Clean Makeup Removing Cleansing Balm,Farmacy,Skincare,36.0,4.4958,6158.0,403801,9.966515
4,Ultra Repair Cream Intense Hydration,First Aid Beauty,Skincare,38.0,4.5200,7539.0,300432,9.816586
5,Facial Cotton,Shiseido,Skincare,13.0,4.8284,2827.0,133163,9.322855
6,Niacinamide 10% + Zinc 1% Oil Control Serum,The Ordinary,Skincare,6.0,4.2439,5778.0,763168,10.266509
7,The True Cream Aqua Bomb,belif,Skincare,38.0,4.4841,7292.0,265050,9.727304
8,Mini Daily Microfoliant Exfoliator,Dermalogica,Skincare,18.0,4.6960,4598.0,110141,9.217875
9,Daily Microfoliant Exfoliator,Dermalogica,Skincare,65.0,4.6960,4598.0,95311,9.131106


In [60]:
recommend_popular(
    category="Makeup"
)

,product_name,brand_name,primary_category,price_usd,rating,reviews,loves_count,popularity_score
2348,Soft Pinch Liquid Blush,Rare Beauty by Selena Gomez,Makeup,23.0,4.5356,4733.0,1401068,10.698580
2349,Radiant Creamy Concealer,NARS,Makeup,32.0,4.3080,12887.0,1153594,10.613841
2351,Cream Lip Stain Liquid Lipstick,SEPHORA COLLECTION,Makeup,15.0,4.3201,11111.0,1029051,10.534097
2352,Gloss Bomb Universal Lip Luminizer,Fenty Beauty by Rihanna,Makeup,21.0,4.6357,12136.0,968317,10.601101
2353,Pro Filt’r Soft Matte Longwear Liquid Foundation,Fenty Beauty by Rihanna,Makeup,40.0,4.0356,16935.0,856497,10.380764
2354,Blush,NARS,Makeup,32.0,4.6643,18127.0,840076,10.564561
2355,Brow Wiz Ultra-Slim Precision Brow Pencil,Anastasia Beverly Hills,Makeup,25.0,4.4056,15885.0,834189,10.469529
2356,Translucent Loose Setting Powder,Laura Mercier,Makeup,43.0,4.5029,9335.0,813497,10.430493
2357,Dior Addict Lip Glow,Dior,Makeup,40.0,4.3517,2084.0,757716,10.192602
2358,Lip Glow Oil,Dior,Makeup,40.0,4.0795,1145.0,684860,9.990436


In [61]:
recommend_popular(
    brand="LANEIGE"
)

,product_name,brand_name,primary_category,price_usd,rating,reviews,loves_count,popularity_score
0,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Skincare,24.0,4.3508,16118.0,1081315,10.610229
13,Lip Glowy Balm,LANEIGE,Skincare,18.0,4.3923,3722.0,471406,9.978005
42,Cream Skin Toner & Moisturizer,LANEIGE,Skincare,33.0,4.4481,1859.0,143173,9.210353
54,Cica Sleeping Mask,LANEIGE,Skincare,34.0,4.6375,971.0,80111,8.853894
176,Berries n' Choco Kisses Set,LANEIGE,Skincare,26.0,4.5954,173.0,71206,8.598533
261,Water Sleeping Mask with Squalane,LANEIGE,Skincare,32.0,4.3623,472.0,78940,8.690473
274,Water Bank Blue Hyaluronic Serum,LANEIGE,Skincare,45.0,4.7355,310.0,24759,8.064820
288,Radian-C Cream with Vitamin C,LANEIGE,Skincare,35.0,4.4913,631.0,45923,8.433125
343,Cream Skin Mist,LANEIGE,Skincare,27.0,4.5739,291.0,43168,8.343572
422,Lip Treatment Balm,LANEIGE,Skincare,25.0,4.2128,719.0,93358,8.788290


In [62]:
recommend_popular(
    price_bucket="Budget"
)

,product_name,brand_name,primary_category,price_usd,rating,reviews,loves_count,popularity_score
0,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Skincare,24.0,4.3508,16118.0,1081315,10.610229
1,AHA 30% + BHA 2% Exfoliating Peeling Solution,The Ordinary,Skincare,9.5,4.5734,3242.0,533877,10.093199
5,Facial Cotton,Shiseido,Skincare,13.0,4.8284,2827.0,133163,9.322855
6,Niacinamide 10% + Zinc 1% Oil Control Serum,The Ordinary,Skincare,6.0,4.2439,5778.0,763168,10.266509
8,Mini Daily Microfoliant Exfoliator,Dermalogica,Skincare,18.0,4.6960,4598.0,110141,9.217875
10,Glycolic Acid 7% Exfoliating Toning Solution,The Ordinary,Skincare,13.0,4.3456,2891.0,542743,10.023286
13,Lip Glowy Balm,LANEIGE,Skincare,18.0,4.3923,3722.0,471406,9.978005
14,Intense Therapy Lip Balm SPF 25,Jack Black,Skincare,10.0,4.6818,3595.0,123439,9.257404
15,Hyaluronic Acid 2% + B5 Hydrating Serum,The Ordinary,Skincare,15.7,4.2133,3788.0,720504,10.180600
17,Lactic Acid 10% + HA 2% Exfoliating Serum,The Ordinary,Skincare,8.9,4.5035,1289.0,295071,9.624275


In [63]:
recommend_popular(
    category="Skincare",
    price_bucket="Affordable",
    top_n=5
)

,product_name,brand_name,primary_category,price_usd,rating,reviews,loves_count,popularity_score
3,Green Clean Makeup Removing Cleansing Balm,Farmacy,Skincare,36.0,4.4958,6158.0,403801,9.966515
4,Ultra Repair Cream Intense Hydration,First Aid Beauty,Skincare,38.0,4.5200,7539.0,300432,9.816586
7,The True Cream Aqua Bomb,belif,Skincare,38.0,4.4841,7292.0,265050,9.727304
12,The True Cream Moisturizing Bomb,belif,Skincare,38.0,4.5631,4523.0,151868,9.369109
16,100 percent Pure Argan Oil,Josie Maran,Skincare,49.0,4.4998,7763.0,134089,9.329425


In [64]:
def recommend_popular(
    category=None,
    brand=None,
    price_bucket=None,
    top_n=10
):

    recommendations = popular_products.copy()

    if category:
        recommendations = recommendations[
            recommendations["primary_category"]
            .str.lower()
            ==
            category.lower()
        ]

    if brand:
        recommendations = recommendations[
            recommendations["brand_name"]
            .str.lower()
            ==
            brand.lower()
        ]

    if price_bucket:
        recommendations = recommendations[
            recommendations["price_bucket"]
            ==
            price_bucket
        ]

    recommendations = recommendations.head(top_n).copy()

    recommendations.insert(
        0,
        "Rank",
        range(1, len(recommendations)+1)
    )

    return recommendations[
        [
            "Rank",
            "product_name",
            "brand_name",
            "primary_category",
            "price_usd",
            "rating",
            "reviews",
            "loves_count",
            "popularity_score"
        ]
    ]

In [65]:
products.to_csv(
    "products_processed_popularity.csv",
    index=False
)

In [66]:
reviews.to_csv(
    "reviews_processed_popularity.csv",
    index=False
)

In [67]:
"text_blob" in products.columns

True

In [68]:
products[["product_name", "text_blob"]].head()

,product_name,text_blob
0,Fragrance Discovery Set,Fragrance Discovery Set 19-69 ['Unisex/ Gender...
1,La Habana Eau de Parfum,La Habana Eau de Parfum 19-69 ['Unisex/ Gender...
2,Rainbow Bar Eau de Parfum,Rainbow Bar Eau de Parfum 19-69 ['Unisex/ Gend...
3,Kasbah Eau de Parfum,Kasbah Eau de Parfum 19-69 ['Unisex/ Genderles...
4,Purple Haze Eau de Parfum,Purple Haze Eau de Parfum 19-69 ['Unisex/ Gend...
